# Modeling

In [1]:
!pip install statsmodels


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

## Ridge Regression

In [5]:
df = pd.read_csv('../datasets/normalized_data/combined_normalized.csv')
df

,Transacted Price ($),Area (SQFT),Unit Price ($ PSF),Dist to CBD in Km,Sale Months Since Sep 2025,Num MRT Within 1km,Num Hawker Within 1km,Num Malls Within 1km,Num Hospitals Within 5km,Num Schools Within 2km,Num Parks Within 1km,Floor_Level_Category,Type_of_Sale_Encoded,Property_Type_Encoded,Market_Segment_Encoded,District
0,-0.211237,-0.460443,0.774350,1.347673,-1.117103,-1.177227,-1.175060,-1.296450,-1.143547,-1.118512,-1.374335,-1.619747,-1.228179,-0.334688,1.124971,1
1,-0.087630,-0.460443,1.240198,1.347673,-1.117103,-1.177227,-1.175060,-1.296450,-1.143547,-1.118512,-1.374335,0.947931,-1.228179,-0.334688,1.124971,1
2,-0.303362,-0.331436,-0.058154,-0.618836,-1.117103,-0.723616,-0.407922,0.640686,0.276567,-0.286674,0.161507,-0.152502,0.814214,2.987854,-0.888912,1
3,-0.100129,-0.442004,1.103809,1.347673,-1.117103,-1.177227,-1.175060,-1.296450,-1.143547,-1.118512,-1.374335,0.214309,-1.228179,-0.334688,1.124971,1
4,1.493910,2.267081,-0.950882,-0.363508,-1.117103,0.183604,-0.407922,0.156402,-0.670176,-0.286674,-0.030473,-1.619747,0.814214,-0.334688,-0.888912,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104128,-0.947937,-0.579614,-0.985079,0.574579,1.673034,0.833925,1.313703,-0.328072,-0.506403,0.980484,-0.613859,1.310748,0.037050,-1.351717,0.000000,28
104129,-0.937078,0.166615,-2.397949,-1.124401,1.673034,-1.632820,-1.555130,-1.985871,2.556176,-1.696390,1.710013,-0.905237,0.037050,0.739800,0.000000,28
104130,-1.023949,-0.888409,-0.204967,0.512166,1.673034,0.340576,-0.120714,-0.328072,-0.506403,-0.506668,-0.355651,2.049409,0.037050,-1.351717,0.000000,28
104131,0.702603,1.504712,-1.513821,0.246256,1.673034,0.833925,1.313703,1.329727,-0.506403,0.980484,-0.872068,-0.905237,0.037050,0.739800,0.000000,28


## Ridge Regression

In [10]:
print("Ridge Regression Modeling for Singapore Private Residential Property Prices")

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("DATA PREPARATION")

df = pd.read_csv('../datasets/normalized_data/combined_normalized.csv')

print(f"Loaded {len(df)} property transactions")
print(f"   Date range: {df['Sale Months Since Sep 2025'].min():.2f} to {df['Sale Months Since Sep 2025'].max():.2f} months")
print(f"   Districts: {df['District'].nunique()}")
print(f"   Features: {len(df.columns)}")

# Check for missing values
missing = df.isnull().sum()
if missing.sum() > 0:
    print(f"\nWARNING: Found {missing.sum()} missing values")
    print(missing[missing > 0])
    df = df.dropna()
    print(f"   After dropping: {len(df)} transactions remain")
else:
    print(f"\nNo missing values found")


print("\nFEATURE ENGINEERING")

location_features = ['Dist to CBD in Km']
amenity_features = [
    'Num MRT Within 1km', 
    'Num Hawker Within 1km', 
    'Num Malls Within 1km', 
    'Num Hospitals Within 5km', 
    'Num Schools Within 2km', 
    'Num Parks Within 1km'
]
property_features = [
    'Area (SQFT)',
    'Floor_Level_Category', 
    'Type_of_Sale_Encoded', 
    'Property_Type_Encoded', 
    'Market_Segment_Encoded'
]
time_features = ['Sale Months Since Sep 2025']

base_features = location_features + amenity_features + property_features + time_features

print(f"\nFeature Summary:")
print(f"   Location features: {len(location_features)}")
print(f"   Amenity features: {len(amenity_features)}")
print(f"   Property features: {len(property_features)}")
print(f"   Time features: {len(time_features)}")

district_dummies = pd.get_dummies(df['District'], prefix='District', drop_first=True)

print(f"District dummies: {len(district_dummies.columns)} (baseline: District {df['District'].min()})")

X_base = df[base_features].copy()
X = pd.concat([X_base, district_dummies], axis=1)
y = df['Transacted Price ($)'].copy()

print(f"   TOTAL FEATURES: {X.shape[1]}")
print(f"   TOTAL SAMPLES: {len(X)}")

print("\n MULTICOLLINEARITY CHECK (VIF)")

# Calculate VIF 
vif_data = pd.DataFrame()
vif_data["Feature"] = base_features

vif_values = []
for i in range(len(base_features)):
    try:
        vif = variance_inflation_factor(X[base_features].values, i)
        vif_values.append(vif)
    except:
        vif_values.append(np.nan)

vif_data["VIF"] = vif_values
vif_data = vif_data.sort_values('VIF', ascending=False)

print(f"\nVariance Inflation Factor (VIF) Analysis:")
print(f"   VIF < 5:  Low multicollinearity")
print(f"   VIF 5-10: Moderate multicollinearity")
print(f"   VIF > 10: High multicollinearity")
print("\n" + "-"*60)
print(vif_data.to_string(index=False))

high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f"\n{len(high_vif)} features have high multicollinearity")
    print(high_vif[['Feature', 'VIF']].to_string(index=False))
    print(f"\n   Ridge regression will help stabilize coefficients!")
else:
    print(f"\nAll features have acceptable VIF values")

print("\nFINDING OPTIMAL RIDGE REGULARIZATION (ALPHA) --> Hyperparameter Tuning")

alphas = np.logspace(-3, 3, 100)  # From 0.001 to 1000

ridge_cv = RidgeCV(alphas=alphas, cv=10, scoring='r2')
ridge_cv.fit(X, y)

optimal_alpha = ridge_cv.alpha_
print(f"\nOptimal Alpha: {optimal_alpha:.6f}")
print(f"   10-Fold CV R²: {ridge_cv.best_score_:.4f}")
print(f"\nUsing ALL {len(X)} samples for analysis (no train/test split)")
print(f"   Cross-validation ensures robust parameter selection")

print("\nFINAL RIDGE REGRESSION MODEL")

# Use optimal alpha from cross-validation
model = Ridge(alpha=optimal_alpha)
model.fit(X, y)

print(f"\n✅ Ridge Regression Model:")
print(f"   Alpha: {optimal_alpha:.6f}")
print(f"   Features: {len(model.coef_)} (all retained)")
print(f"   Samples: {len(X)}")

print("\nMODEL PERFORMANCE METRICS")

# Predictions on all data
y_pred = model.predict(X)

# Calculate metrics
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
mae = mean_absolute_error(y, y_pred)

def mape(y_true, y_pred):
    y_true_actual = y_true * df['Transacted Price ($)'].std() + df['Transacted Price ($)'].mean()
    y_pred_actual = y_pred * df['Transacted Price ($)'].std() + df['Transacted Price ($)'].mean()
    
    mask = y_true_actual != 0
    return np.mean(np.abs((y_true_actual[mask] - y_pred_actual[mask]) / y_true_actual[mask])) * 100


print(f"\nMODEL FIT METRICS (All Data):")
print(f"   R² Score:  {r2:.4f}  ({r2*100:.2f}% of variance explained)")
print(f"   RMSE:      {rmse:.4f} (normalized)")
print(f"   MAE:       {mae:.4f} (normalized)")

print(f"\n10-Fold Cross-Validation (Robustness Check):")
cv_r2_scores = cross_val_score(model, X, y, cv=10, scoring='r2')
cv_mae_scores = -cross_val_score(model, X, y, cv=10, scoring='neg_mean_absolute_error')

print(f"   R² Score:")
print(f"      Mean: {cv_r2_scores.mean():.4f}")
print(f"      Std:  {cv_r2_scores.std():.4f}")
print(f"      Range: [{cv_r2_scores.min():.4f}, {cv_r2_scores.max():.4f}]")

print(f"\n   MAE:")
print(f"      Mean: {cv_mae_scores.mean():.4f}")
print(f"      Std:  {cv_mae_scores.std():.4f}")

# Interpret the gap
cv_gap = r2 - cv_r2_scores.mean()
print(f"\nMODEL STABILITY:")
print(f"   R² Gap (Full Model - CV Mean): {cv_gap:.4f}")
if cv_gap > 0.05:
    print(f"   Moderate optimism - model fits training data better than CV suggests")
elif cv_gap > 0.02:
    print(f"   Good - small optimism is normal")
else:
    print(f"   Excellent - very stable model!")

print(f"\nInterpretation:")
print(f"   • R² = {r2:.4f} means the model explains {r2*100:.1f}% of price variation")
print(f"   • Cross-validation R² = {cv_r2_scores.mean():.4f} suggests similar performance")
print(f"     on unseen data folds")

# Residual analysis
residuals = y - y_pred
print(f"\nRESIDUAL ANALYSIS:")
print(f"   Mean residual: {residuals.mean():.6f} (should be ~0)")
print(f"   Std residual:  {residuals.std():.4f}")
print(f"   Min residual:  {residuals.min():.4f}")
print(f"   Max residual:  {residuals.max():.4f}")

Ridge Regression Modeling for Singapore Private Residential Property Prices
DATA PREPARATION
Loaded 104133 property transactions
   Date range: -4.39 to 2.88 months
   Districts: 27
   Features: 16

No missing values found

FEATURE ENGINEERING

Feature Summary:
   Location features: 1
   Amenity features: 6
   Property features: 5
   Time features: 1
District dummies: 26 (baseline: District 1)
   TOTAL FEATURES: 39
   TOTAL SAMPLES: 104133

 MULTICOLLINEARITY CHECK (VIF)

Variance Inflation Factor (VIF) Analysis:
   VIF < 5:  Low multicollinearity
   VIF 5-10: Moderate multicollinearity
   VIF > 10: High multicollinearity

------------------------------------------------------------
                   Feature      VIF
         Dist to CBD in Km 1.743033
  Num Hospitals Within 5km 1.526258
    Market_Segment_Encoded 1.262860
      Num Malls Within 1km 1.251256
        Num MRT Within 1km 1.219675
    Num Schools Within 2km 1.203934
      Type_of_Sale_Encoded 1.185563
     Num Hawker With

In [11]:
print("\nCOEFFICIENT ANALYSIS")

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)

district_coefs = coef_df[coef_df['Feature'].str.startswith('District_')].copy()
other_coefs = coef_df[~coef_df['Feature'].str.startswith('District_')].copy()


print("\nBASE FEATURE COEFFICIENTS (Non-District)")
print(other_coefs.to_string(index=False))

print("\nINTERPRETATION:")
print("   • Positive coefficient → Increases property price")
print("   • Negative coefficient → Decreases property price")
print("   • Magnitude indicates strength of relationship")
print("   • All features are normalized (mean=0, std=1) for fair comparison")

print("\nTOP POSITIVE EFFECTS:")
top_positive = other_coefs.nlargest(3, 'Coefficient')
for idx, row in top_positive.iterrows():
    print(f"   • {row['Feature']}: {row['Coefficient']:.4f}")
    
print("\nTOP NEGATIVE EFFECTS:")
top_negative = other_coefs.nsmallest(3, 'Coefficient')
for idx, row in top_negative.iterrows():
    print(f"   • {row['Feature']}: {row['Coefficient']:.4f}")


print("\nDISTRICT COEFFICIENTS (vs. Baseline District)")
print("-"*80)

baseline_district = df['District'].min()
print(f"\nBaseline District: District {baseline_district}")
print(f"   All other districts are compared to this baseline\n")

print(district_coefs.to_string(index=False))

print("\nINTERPRETATION:")
print(f"   • Positive coefficient → More expensive than District {baseline_district}")
print(f"   • Negative coefficient → Less expensive than District {baseline_district}")
print("   • Coefficient shows the price premium/discount")

# Highlight premium 
print("\nPREMIUM DISTRICTS (Highest Price Premium):")
premium_districts = district_coefs.nlargest(5, 'Coefficient')
for idx, row in premium_districts.iterrows():
    district_num = row['Feature'].replace('District_', '')
    print(f"   • District {district_num}: +{row['Coefficient']:.4f} (vs. baseline)")

print("\nBUDGET DISTRICTS (Largest Price Discount):")
budget_districts = district_coefs.nsmallest(5, 'Coefficient')
for idx, row in budget_districts.iterrows():
    district_num = row['Feature'].replace('District_', '')
    print(f"   • District {district_num}: {row['Coefficient']:.4f} (vs. baseline)")


print("\nGENERATING COEFFICIENT VISUALIZATIONS")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

other_coefs_sorted = other_coefs.sort_values('Coefficient')
colors = ['red' if x < 0 else 'green' for x in other_coefs_sorted['Coefficient']]

ax1.barh(range(len(other_coefs_sorted)), other_coefs_sorted['Coefficient'], color=colors, alpha=0.7)
ax1.set_yticks(range(len(other_coefs_sorted)))
ax1.set_yticklabels(other_coefs_sorted['Feature'], fontsize=9)
ax1.set_xlabel('Coefficient Value', fontsize=12, fontweight='bold')
ax1.set_title('Base Feature Coefficients\n(Green = Positive Effect, Red = Negative Effect)', 
              fontsize=13, fontweight='bold')
ax1.axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax1.grid(axis='x', alpha=0.3)

district_coefs_sorted = district_coefs.sort_values('Coefficient')
colors_district = ['red' if x < 0 else 'green' for x in district_coefs_sorted['Coefficient']]
district_labels = [f"D{x.replace('District_', '')}" for x in district_coefs_sorted['Feature']]

ax2.barh(range(len(district_coefs_sorted)), district_coefs_sorted['Coefficient'], 
         color=colors_district, alpha=0.7)
ax2.set_yticks(range(len(district_coefs_sorted)))
ax2.set_yticklabels(district_labels, fontsize=8)
ax2.set_xlabel('Coefficient Value (vs. Baseline)', fontsize=12, fontweight='bold')
ax2.set_title(f'District Price Effects (vs. District {baseline_district})\n(Green = Premium, Red = Discount)', 
              fontsize=13, fontweight='bold')
ax2.axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('./outputs/ridge/coefficient_analysis.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: ./outputs/ridge/coefficient_analysis.png")

# Feature Importance by Category
print("\nGenerating feature importance by category...")

feature_categories = {
    'Location': location_features,
    'Amenities': amenity_features,
    'Property': property_features,
    'Time': time_features
}

category_importance = {}
for category, features in feature_categories.items():
    coefs_in_category = other_coefs[other_coefs['Feature'].isin(features)]['Coefficient'].abs().sum()
    category_importance[category] = coefs_in_category

category_df = pd.DataFrame(list(category_importance.items()), 
                          columns=['Category', 'Total Absolute Coefficient'])
category_df = category_df.sort_values('Total Absolute Coefficient', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(category_df['Category'], category_df['Total Absolute Coefficient'], 
              color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'], alpha=0.7)
ax.set_ylabel('Total Absolute Coefficient', fontsize=12, fontweight='bold')
ax.set_xlabel('Feature Category', fontsize=12, fontweight='bold')
ax.set_title('Feature Category Importance\n(Sum of Absolute Coefficients)', 
             fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}',
            ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('./outputs/ridge/category_importance.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: ./outputs/ridge/category_importance.png")

print("\n" + category_df.to_string(index=False))

print("\nCOEFFICIENT SUMMARY")

summary_stats = pd.DataFrame({
    'Category': ['Base Features', 'District Effects', 'All Features'],
    'Count': [len(other_coefs), len(district_coefs), len(coef_df)],
    'Positive': [
        (other_coefs['Coefficient'] > 0).sum(),
        (district_coefs['Coefficient'] > 0).sum(),
        (coef_df['Coefficient'] > 0).sum()
    ],
    'Negative': [
        (other_coefs['Coefficient'] < 0).sum(),
        (district_coefs['Coefficient'] < 0).sum(),
        (coef_df['Coefficient'] < 0).sum()
    ],
    'Mean Abs Coef': [
        other_coefs['Coefficient'].abs().mean(),
        district_coefs['Coefficient'].abs().mean(),
        coef_df['Coefficient'].abs().mean()
    ],
    'Max Positive': [
        other_coefs['Coefficient'].max(),
        district_coefs['Coefficient'].max(),
        coef_df['Coefficient'].max()
    ],
    'Max Negative': [
        other_coefs['Coefficient'].min(),
        district_coefs['Coefficient'].min(),
        coef_df['Coefficient'].min()
    ]
})

print("\n" + summary_stats.to_string(index=False))


print("\nCOEFFICIENT ANALYSIS COMPLETE")



COEFFICIENT ANALYSIS

BASE FEATURE COEFFICIENTS (Non-District)
                   Feature  Coefficient
               Area (SQFT)     0.840707
      Floor_Level_Category     0.061090
      Num Parks Within 1km     0.022899
  Num Hospitals Within 5km     0.012625
      Num Malls Within 1km     0.008005
     Num Hawker Within 1km    -0.001813
    Market_Segment_Encoded    -0.013912
    Num Schools Within 2km    -0.015078
     Property_Type_Encoded    -0.019842
        Num MRT Within 1km    -0.027030
         Dist to CBD in Km    -0.036537
Sale Months Since Sep 2025    -0.233705
      Type_of_Sale_Encoded    -0.329484

INTERPRETATION:
   • Positive coefficient → Increases property price
   • Negative coefficient → Decreases property price
   • Magnitude indicates strength of relationship
   • All features are normalized (mean=0, std=1) for fair comparison

TOP POSITIVE EFFECTS:
   • Area (SQFT): 0.8407
   • Floor_Level_Category: 0.0611
   • Num Parks Within 1km: 0.0229

TOP NEGATIVE EFFE

## Ridge Regression Per District

In [12]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("RIDGE REGRESSION ANALYSIS - PER DISTRICT")
print("="*80)


print("\nDATA LOADING")

# Load the FULL dataset with all districts
df_full = pd.read_csv('../datasets/normalized_data/combined_normalized.csv')

print(f"Loaded {len(df_full)} total property transactions")
print(f"   Total features: {len(df_full.columns)}")

# Check for missing values
missing = df_full.isnull().sum()
if missing.sum() > 0:
    print(f"\nFound {missing.sum()} missing values")
    df_full = df_full.dropna()
    print(f"   After dropping: {len(df_full)} transactions remain")
else:
    print(f"\nNo missing values found")

# Extract postal district from the data
if 'Postal District' in df_full.columns:
    postal_district_col = 'Postal District'
elif 'District' in df_full.columns:
    postal_district_col = 'District'
else:
    # If not available, create from address or other info
    print("No district column found - creating from data...")
    # You may need to adjust this based on your data structure

# Get unique districts
districts = sorted(df_full[postal_district_col].unique())
print(f"\nFound {len(districts)} unique postal districts:")
print(f"   Districts: {districts}")

print("\n FEATURE DEFINITION")

# Define feature groups (EXCLUDING district dummies - we're doing per-district analysis)
location_features = ['Dist to CBD in Km']
amenity_features = [
    'Num MRT Within 1km', 
    'Num Hawker Within 1km', 
    'Num Malls Within 1km', 
    'Num Hospitals Within 5km', 
    'Num Schools Within 2km', 
    'Num Parks Within 1km'
]
property_features = [
    'Area (SQFT)',
    'Floor_Level_Category', 
    'Type_of_Sale_Encoded', 
    'Property_Type_Encoded', 
    'Market_Segment_Encoded'
]
time_features = ['Sale Months Since Sep 2025']

# Combine all features
all_features = location_features + amenity_features + property_features + time_features

print(f"\nFeature Summary:")
print(f"   Location features: {len(location_features)}")
print(f"   Amenity features: {len(amenity_features)}")
print(f"   Property features: {len(property_features)}")
print(f"   Time features: {len(time_features)}")
print(f"   TOTAL FEATURES: {len(all_features)}")


print("\nRIDGE REGRESSION - DISTRICT BY DISTRICT")

# Storage for results
district_results = []
district_models = {}
district_coefficients = {}

# Analyze each district
for district in districts:
    print(f"\n{'='*80}")
    print(f"ANALYZING DISTRICT {district}")
    print(f"{'='*80}")
    
    # Filter data for this district
    df_district = df_full[df_full[postal_district_col] == district].copy()
    
    print(f"\nDistrict {district} Statistics:")
    print(f"   Total transactions: {len(df_district)}")
    print(f"   Price range: ${df_district['Transacted Price ($)'].min():,.0f} - ${df_district['Transacted Price ($)'].max():,.0f}")
    print(f"   Median price: ${df_district['Transacted Price ($)'].median():,.0f}")
    
    # Check if we have enough samples
    if len(df_district) < 50:
        print(f"Only {len(df_district)} samples - skipping (need at least 50)")
        continue
    
    # Prepare features and target
    X_district = df_district[all_features].copy()
    y_district = df_district['Transacted Price ($)'].copy()
    
    print(f"\nFeature Matrix:")
    print(f"   Shape: {X_district.shape}")
    print(f"   Target shape: {y_district.shape}")
    
    # Find optimal alpha using cross-validation
    print(f"\nFinding optimal alpha...")
    alphas = np.logspace(-3, 3, 100)
    
    try:
        ridge_cv = RidgeCV(alphas=alphas, cv=min(10, len(df_district)//10), scoring='r2')
        ridge_cv.fit(X_district, y_district)
        
        optimal_alpha = ridge_cv.alpha_
        cv_r2 = ridge_cv.best_score_
        
        print(f"Optimal alpha: {optimal_alpha:.6f}")
        print(f"CV R² score: {cv_r2:.4f}")
        
        # Fit final model with optimal alpha
        model = Ridge(alpha=optimal_alpha)
        model.fit(X_district, y_district)
        
        # Make predictions
        y_pred = model.predict(X_district)
        
        # Calculate metrics
        r2 = r2_score(y_district, y_pred)
        rmse = np.sqrt(mean_squared_error(y_district, y_pred))
        mae = mean_absolute_error(y_district, y_pred)
        
        # Calculate MAPE
        mape_value = np.mean(np.abs((y_district - y_pred) / y_district)) * 100
        
        print(f"\nModel Performance:")
        print(f"   R² Score:  {r2:.4f}")
        print(f"   RMSE:      ${rmse:,.0f}")
        print(f"   MAE:       ${mae:,.0f}")
        print(f"   MAPE:      {mape_value:.2f}%")
        
        # Store results
        district_results.append({
            'District': district,
            'Num_Samples': len(df_district),
            'Optimal_Alpha': optimal_alpha,
            'CV_R2': cv_r2,
            'R2_Score': r2,
            'RMSE': rmse,
            'MAE': mae,
            'MAPE': mape_value,
            'Median_Price': df_district['Transacted Price ($)'].median()
        })
        
        # Store model and coefficients
        district_models[district] = model
        
        coef_dict = dict(zip(all_features, model.coef_))
        coef_dict['Intercept'] = model.intercept_
        district_coefficients[district] = coef_dict
        
    except Exception as e:
        print(f"Error fitting model: {str(e)}")
        continue


print("\nCROSS-DISTRICT COMPARISON")

# Create results DataFrame
results_df = pd.DataFrame(district_results)
results_df = results_df.sort_values('R2_Score', ascending=False)

print("\nMODEL PERFORMANCE SUMMARY (Ranked by R²):")
print(results_df.to_string(index=False))

print("\nGENERATING VISUALIZATIONS")

# 1. R² Comparison across districts
plt.figure(figsize=(14, 6))
bars = plt.bar(results_df['District'].astype(str), results_df['R2_Score'], 
               color='steelblue', edgecolor='black', linewidth=1.5)

for i, bar in enumerate(bars):
    r2 = results_df.iloc[i]['R2_Score']
    if r2 >= 0.8:
        bar.set_color('green')
    elif r2 >= 0.6:
        bar.set_color('orange')
    else:
        bar.set_color('red')

plt.axhline(y=results_df['R2_Score'].mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean R² = {results_df["R2_Score"].mean():.3f}')
plt.xlabel('Postal District', fontsize=12, fontweight='bold')
plt.ylabel('R² Score', fontsize=12, fontweight='bold')
plt.title('Ridge Regression Performance by District', fontsize=14, fontweight='bold')
plt.ylim(0, 1)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./outputs/ridge/district_r2_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: ./outputs/ridge/district_r2_comparison.png")

coef_df = pd.DataFrame(district_coefficients).T
coef_df = coef_df[all_features] 

plt.figure(figsize=(14, max(8, len(districts)*0.5)))
sns.heatmap(coef_df, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Coefficient Value'}, linewidths=0.5)
plt.xlabel('Features', fontsize=12, fontweight='bold')
plt.ylabel('District', fontsize=12, fontweight='bold')
plt.title('Ridge Regression Coefficients Across Districts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./outputs/ridge/coefficient_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"Saved: ./outputs/ridge/coefficient_heatmap.png")

print(f"\nTop 5 Most Important Features by District:")
for district in districts:
    if district in district_coefficients:
        coefs = district_coefficients[district]
        coefs_features = {k: v for k, v in coefs.items() if k != 'Intercept'}
        # Sort by absolute value
        top_features = sorted(coefs_features.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
        
        print(f"\n   District {district}:")
        for feature, coef in top_features:
            print(f"      {feature:30s}: {coef:+.4f}")

print("\nDISTRICT-LEVEL RIDGE REGRESSION ANALYSIS COMPLETE!")

RIDGE REGRESSION ANALYSIS - PER DISTRICT

DATA LOADING
Loaded 104133 total property transactions
   Total features: 16

No missing values found

Found 27 unique postal districts:
   Districts: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 27, 28]

 FEATURE DEFINITION

Feature Summary:
   Location features: 1
   Amenity features: 6
   Property features: 5
   Time features: 1
   TOTAL FEATURES: 13

RIDGE REGRESSION - DISTRICT BY DISTRICT

ANALYZING DISTRICT 1

District 1 Statistics:
   Total transactions: 1638
   Price range: $-1 - $13
   Median price: $-0

Feature Matrix:
   Shape: (1638, 13)
   Target shape: (1638,)

Finding optimal alpha...
Optimal alpha: 3.274549
CV R² score: 0.9093

Model Performance:
   R² Score:  0.9132
   RMSE:      $0
   MAE:       $0
   MAPE:      159.21%

ANALYZING DISTRICT 2

District 2 Statistics:
   Total transactions: 1207
   Price range: $-1 - $29
   Median price: $-0

Feature Matrix:
   Shape: (1207, 13)
   T